# Zn ExpertXAS: evaluate 310–500–750

Compare the saved **310–500–750 ExpertXAS** with a nine-element **UniversalXAS** and its Zn-tuned version. All three use the same frozen encoder features. The universal and Zn-tuning sweeps select checkpoints using Zn validation MSE.

The first cells check the checkpoint, encoder, features, dataset, and provenance. Set `RUN_HEAD_TRAINING = True` to run the two new head sweeps. Existing completed trials are reused only when their provenance matches. Missing files stop execution. This exploratory dataset uses unscaled targets and is not directly comparable with the paper.


In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import sys
from unittest.mock import patch

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "omnixas").is_dir())
sys.path.insert(0, str(REPO_ROOT))
DATA_ROOT = Path(os.environ.get("OMNIXAS_DATA_ROOT", REPO_ROOT.parent / "OmniXAS_data")).expanduser().resolve()
DATASET_ROOT = DATA_ROOT / "figshare_141_splits_unscaled_zn9"
RAW_ROOT = DATA_ROOT / "figshare"
RUN_ROOT = REPO_ROOT / "output" / "training" / "zn_expert_sweep"
ENCODER_RUN = RUN_ROOT / "zn9_universal_encoder"
FEATURES_DIR = ENCODER_RUN / "features"
HEAD_DIR = RUN_ROOT / "expert_sweep_tens_200" / "heads" / "310x500x750"
OUTPUT_DIR = RUN_ROOT / "evaluation_310x500x750"
DIMS = [310, 500, 750]
SPLITS = ("train", "val", "test")
ENERGY_GRID = np.linspace(9659.0, 9694.0, 141)
SIGMA_EV = 0.5
BATCH_SIZE = 4096

def sha256(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

required = [HEAD_DIR / "best.pt", HEAD_DIR / "metrics.json", ENCODER_RUN / "best_encoder.ckpt",
            ENCODER_RUN / "RUN_COMPLETE.json", ENCODER_RUN / "notebook_dataset_sha256.txt",
            DATASET_ROOT / "dataset.json"]
for split in SPLITS:
    required += [FEATURES_DIR / f"Zn_FEFF_{split}_{suffix}.txt" for suffix in ("X", "y", "ids")]
    required += [DATASET_ROOT / "material_id_and_site" / f"Zn_FEFF_{split}.txt",
                 DATASET_ROOT / "ml_data" / f"Zn_FEFF_{split}_y.txt"]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required local artifacts:\n" + "\n".join(missing))
print("All required local artifacts found.")


## Verify the checkpoint and row order

The head must match its recorded checkpoint, encoder, dataset, and training and validation feature hashes. Test IDs and targets must match the canonical dataset. These checks do not independently prove that test feature rows were generated in the correct order.


In [ ]:
import torch
from omnixas.model.xasblock import XASBlock

record = json.loads((HEAD_DIR / "metrics.json").read_text(encoding="utf-8"))
provenance = record.get("provenance", {})
if record.get("hidden_dims") != "310-500-750" or record.get("checkpoint_sha256") != sha256(HEAD_DIR / "best.pt"):
    raise ValueError("The head checkpoint or its architecture does not match its recorded provenance.")
if provenance.get("encoder_sha256") != sha256(ENCODER_RUN / "best_encoder.ckpt"):
    raise ValueError("The saved head and encoder do not match.")
dataset_hash = sha256(DATASET_ROOT / "dataset.json")
if provenance.get("dataset_sha256") != dataset_hash or (ENCODER_RUN / "notebook_dataset_sha256.txt").read_text().strip() != dataset_hash:
    raise ValueError("The head, encoder, and canonical dataset provenance do not match.")
if json.loads((ENCODER_RUN / "RUN_COMPLETE.json").read_text()).get("stage") != "encoder_only":
    raise ValueError("The encoder run is not marked complete.")
for split in ("train", "val"):
    if provenance.get("feature_sha256", {}).get(split) != sha256(FEATURES_DIR / f"Zn_FEFF_{split}_X.txt"):
        raise ValueError(f"The {split} features do not match the head's recorded features.")

parts = {}
for split in SPLITS:
    prefix = f"Zn_FEFF_{split}"
    X = np.loadtxt(FEATURES_DIR / f"{prefix}_X.txt", dtype=np.float32, ndmin=2)
    y = np.loadtxt(FEATURES_DIR / f"{prefix}_y.txt", dtype=np.float32, ndmin=2)
    ids = (FEATURES_DIR / f"{prefix}_ids.txt").read_text(encoding="utf-8").splitlines()
    canonical_ids = (DATASET_ROOT / "material_id_and_site" / f"{prefix}.txt").read_text(encoding="utf-8-sig").splitlines()
    canonical_y = np.loadtxt(DATASET_ROOT / "ml_data" / f"{prefix}_y.txt", dtype=np.float32, ndmin=2)
    if X.shape != (len(ids), 64) or y.shape != (len(ids), 141) or not np.isfinite(X).all() or not np.isfinite(y).all():
        raise ValueError(f"Invalid feature or spectrum dimensions for {split}.")
    if ids != canonical_ids or not np.array_equal(y, canonical_y) or len(ids) != len(set(ids)):
        raise ValueError(f"ID or target mismatch for {split}.")
    parts[split] = (ids, X, y)
material_sets = [{value.rsplit("_", 1)[0] for value in parts[split][0]} for split in SPLITS]
if any(material_sets[i] & material_sets[j] for i in range(3) for j in range(i + 1, 3)):
    raise ValueError("A Zn material appears in multiple splits.")

state = torch.load(HEAD_DIR / "best.pt", map_location="cpu", weights_only=False)
if state.get("hidden_dims") != DIMS or float(state.get("dropout", -1)) != 0.05:
    raise ValueError("Checkpoint is not the expected 310–500–750 head.")
with patch.object(XASBlock, "DROPOUT", float(state["dropout"])):
    head = XASBlock(64, DIMS, 141)
head.load_state_dict(state["state_dict"], strict=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
head.to(device).eval()
print(f"Verified 310–500–750 head: {sum(p.numel() for p in head.parameters()):,} trainable parameters; device: {device}.")


## Train UniversalXAS and tune it on Zn

All nine elements share the existing frozen encoder. The existing feature-split loader reads their 64D features. The balancing method keeps every training row and oversamples smaller elements to the largest element's row count.

The universal head uses widths **500–500–550** and the existing fixed loss-balancing power of 1.5. Its 100 trials vary learning rate, dropout, and learning-rate schedule. The best universal checkpoint is selected by **Zn validation median MSE**. The 300 Zn fine-tuning trials start from that checkpoint and vary the same settings. If none improves Zn validation, the unchanged universal checkpoint is retained and labeled unchanged.

Training is off by default. Set `RUN_HEAD_TRAINING = True` to run the sweeps. This can take substantial GPU time. Completed trials are saved separately and checked before reuse. Test results and the iodine outlier must not select a checkpoint. The test set has already been inspected, so this comparison is exploratory.


In [ ]:
import random
from omnixas.data.feff_graph import load_feature_split

RUN_HEAD_TRAINING = False  # Set True on the training machine to run missing trials.
UNIVERSAL_TRIALS, TUNED_TRIALS = 100, 300
UNIVERSAL_EPOCHS, TUNED_EPOCHS, PATIENCE = 800, 1000, 60
HEAD_WIDTHS = [500, 500, 550]
TASKS = [f"{element}_FEFF" for element in ("Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn")]
SWEEP_ROOT = RUN_ROOT / "universal_zn_comparison"

# Use the project's feature loader, then check the original dataset and all ID ordering.
feature_splits, feature_hashes, memberships = {}, {}, {}
for task in TASKS:
    split_data = load_feature_split(FEATURES_DIR, task)
    feature_splits[task] = split_data
    for split in SPLITS:
        part = getattr(split_data, split)
        prefix = f"{task}_{split}"
        ids = (FEATURES_DIR / f"{prefix}_ids.txt").read_text(encoding="utf-8").splitlines()
        canonical_ids = (DATASET_ROOT / "material_id_and_site" / f"{prefix}.txt").read_text(encoding="utf-8-sig").splitlines()
        canonical_y = np.loadtxt(DATASET_ROOT / "ml_data" / f"{prefix}_y.txt", dtype=np.float32, ndmin=2)
        if part.X.shape != (len(ids), 64) or part.y.shape != (len(ids), 141) or not np.isfinite(part.X).all() or not np.isfinite(part.y).all():
            raise ValueError(f"Invalid features or targets: {prefix}")
        if ids != canonical_ids or len(set(ids)) != len(ids) or not np.array_equal(part.y, canonical_y):
            raise ValueError(f"Feature ID or target mismatch: {prefix}")
        for value in ids:
            mid = value.rsplit("_", 1)[0]
            if memberships.setdefault(mid, split) != split:
                raise ValueError(f"Material occurs in different splits: {mid}")
        feature_hashes[prefix] = {"X": sha256(FEATURES_DIR / f"{prefix}_X.txt"),
                                  "y": sha256(FEATURES_DIR / f"{prefix}_y.txt"),
                                  "ids": sha256(FEATURES_DIR / f"{prefix}_ids.txt"),
                                  "canonical_y": sha256(DATASET_ROOT / "ml_data" / f"{prefix}_y.txt"),
                                  "canonical_ids": sha256(DATASET_ROOT / "material_id_and_site" / f"{prefix}.txt")}

# Same keep-all-and-oversample balancing used in the Zn comparison notebook.
rng_balance = np.random.default_rng(42)
n_per_task = max(len(feature_splits[t].train.X) for t in TASKS)
balanced_X, balanced_y, balanced_tasks, train_baselines = [], [], [], []
for task_index, task in enumerate(TASKS):
    part = feature_splits[task].train
    indices = np.r_[np.arange(len(part.X)), rng_balance.integers(len(part.X), size=n_per_task - len(part.X))]
    rng_balance.shuffle(indices)
    balanced_X.append(part.X[indices])
    balanced_y.append(part.y[indices])
    balanced_tasks.append(np.full(n_per_task, task_index, dtype=np.int64))
    train_baselines.append(float(np.median(((part.y - part.y.mean(axis=0)) ** 2).mean(axis=1))))
universal_tasks = np.concatenate(balanced_tasks)
universal_X = np.concatenate(balanced_X)
universal_y = np.concatenate(balanced_y)
zn_val_X, zn_val_y = parts["val"][1:]
zn_train_X, zn_train_y = parts["train"][1:]
identity = {"encoder_sha256": sha256(ENCODER_RUN / "best_encoder.ckpt"), "dataset_sha256": dataset_hash,
            "features_sha256": feature_hashes, "widths": HEAD_WIDTHS, "balance": "keep every row and oversample to largest element", "balance_seed": 42,
            "batch_size": BATCH_SIZE, "patience": PATIENCE, "universal_loss_balance_power": 1.5, "selection": "Zn validation median MSE"}


def make_head(dropout, source=None):
    with patch.object(XASBlock, "DROPOUT", dropout):
        model = XASBlock(64, HEAD_WIDTHS, 141).to(device)
    if source is not None:
        model.load_state_dict(source, strict=True)
    return model


def predict_head(model, X):
    model.eval()
    with torch.inference_mode():
        return torch.cat([model(batch.to(device)).cpu() for batch in torch.from_numpy(X).split(BATCH_SIZE)]).numpy()


def train_trial(path, X, y, hp, max_epochs, source=None, source_hash=None, task_ids=None):
    trial_config = {"identity": identity, "hyperparameters": hp, "max_epochs": max_epochs, "source_sha256": source_hash}
    fingerprint = hashlib.sha256(json.dumps(trial_config, sort_keys=True).encode()).hexdigest()
    checkpoint, result_path = path / "best.pt", path / "metrics.json"
    if checkpoint.is_file() and result_path.is_file():
        result = json.loads(result_path.read_text(encoding="utf-8"))
        if result.get("fingerprint") != fingerprint or result.get("checkpoint_sha256") != sha256(checkpoint):
            raise ValueError(f"Trial provenance mismatch: {path}")
        return result
    if not RUN_HEAD_TRAINING:
        raise FileNotFoundError(f"Missing completed trial {path}. Set RUN_HEAD_TRAINING = True to train it.")
    if path.exists():
        raise RuntimeError(f"Incomplete trial directory: {path}. Choose a new run directory or remove the incomplete trial.")
    path.mkdir(parents=True)
    random.seed(hp["seed"]); np.random.seed(hp["seed"]); torch.manual_seed(hp["seed"])
    if device.type == "cuda":
        torch.cuda.manual_seed_all(hp["seed"])
    model = make_head(hp["dropout"], source)
    Xt, yt = torch.as_tensor(X, device=device), torch.as_tensor(y, device=device)
    vXt, vyt = torch.as_tensor(zn_val_X, device=device), torch.as_tensor(zn_val_y, device=device)
    tasks_tensor = None if task_ids is None else torch.as_tensor(task_ids, dtype=torch.long, device=device)
    bases = torch.as_tensor(train_baselines, dtype=torch.float32, device=device).clamp_min(1e-12)
    def validation_median():
        model.eval()
        with torch.inference_mode():
            return float(torch.quantile((model(vXt) - vyt).square().mean(1), 0.5))
    optimizer = torch.optim.Adam(model.parameters(), lr=hp["lr"], fused=device.type == "cuda")
    scheduler = (torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=16, min_lr=1e-6)
                 if hp["schedule"] == "plateau" else torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=500, eta_min=1e-6))
    best, best_epoch, stale, epochs_run = float("inf"), -1, 0, 0
    if source is not None:
        best = validation_median()
        torch.save({"state_dict": model.state_dict(), "dropout": hp["dropout"], "widths": HEAD_WIDTHS}, checkpoint)
        if hp["schedule"] == "plateau":
            scheduler.step(best)
    for epoch in range(max_epochs):
        model.train()
        batches = list(torch.randperm(len(Xt), device=device).split(BATCH_SIZE))
        if len(batches) > 1 and len(batches[-1]) == 1:
            batches[-2:] = [torch.cat(batches[-2:])]
        for index in batches:
            optimizer.zero_grad(set_to_none=True)
            row_errors = (model(Xt[index]) - yt[index]).square().mean(1)
            if tasks_tensor is None:
                loss = row_errors.mean()
            else:
                batch_tasks = tasks_tensor[index]
                loss = torch.stack([row_errors[batch_tasks == task].mean() / bases[task].pow(1.5)
                                    for task in torch.unique(batch_tasks)]).mean()
            loss.backward()
            optimizer.step()
        metric = validation_median()
        if not np.isfinite(metric):
            raise ValueError(f"Non-finite Zn validation metric: {path}")
        epochs_run = epoch + 1
        if metric < best:
            best, best_epoch, stale = metric, epoch, 0
            torch.save({"state_dict": model.state_dict(), "dropout": hp["dropout"], "widths": HEAD_WIDTHS}, checkpoint)
        else:
            stale += 1
        scheduler.step(metric) if hp["schedule"] == "plateau" else scheduler.step()
        if stale >= PATIENCE:
            break
    if not checkpoint.is_file():
        raise RuntimeError(f"No checkpoint saved for {path}")
    result = {"fingerprint": fingerprint, "config": trial_config, "checkpoint_sha256": sha256(checkpoint),
              "zn_val_median_mse": best, "best_epoch": best_epoch, "epochs_run": epochs_run,
              "selection": "unchanged_universal" if source is not None and best_epoch == -1 else "trained"}
    result_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
    return result


def trial_settings(count, seed, max_lr):
    rng = np.random.default_rng(seed)
    return [{"lr": float(np.exp(rng.uniform(np.log(1e-5), np.log(max_lr)))),
             "dropout": float(rng.choice([0.0, 0.05, 0.10, 0.20])),
             "schedule": str(rng.choice(["plateau", "cosine"])), "seed": int(rng.integers(1_000_000))}
            for _ in range(count)]

universal_results = []
for trial, hp in enumerate(trial_settings(UNIVERSAL_TRIALS, 1042, 1e-2), 1):
    path = SWEEP_ROOT / "universal" / f"trial_{trial:03d}"
    result = train_trial(path, universal_X, universal_y, hp, UNIVERSAL_EPOCHS, task_ids=universal_tasks)
    universal_results.append((result["zn_val_median_mse"], path, result))
    print(f"Universal {trial}/{UNIVERSAL_TRIALS}: Zn validation median MSE {result['zn_val_median_mse']:.6g}", flush=True)
_, universal_path, universal_result = min(universal_results, key=lambda item: item[0])
universal_checkpoint = torch.load(universal_path / "best.pt", map_location="cpu", weights_only=False)
universal_head = make_head(universal_checkpoint["dropout"], universal_checkpoint["state_dict"])
universal_hash = sha256(universal_path / "best.pt")

tuned_results = []
for trial, hp in enumerate(trial_settings(TUNED_TRIALS, 2042, 5e-3), 1):
    path = SWEEP_ROOT / "tuned_zn" / f"trial_{trial:03d}"
    result = train_trial(path, zn_train_X, zn_train_y, hp, TUNED_EPOCHS,
                         source=universal_checkpoint["state_dict"], source_hash=universal_hash)
    tuned_results.append((result["zn_val_median_mse"], path, result))
    print(f"Tuned Zn {trial}/{TUNED_TRIALS}: validation median MSE {result['zn_val_median_mse']:.6g}", flush=True)
_, tuned_path, tuned_result = min(tuned_results, key=lambda item: item[0])
tuned_checkpoint = torch.load(tuned_path / "best.pt", map_location="cpu", weights_only=False)
tuned_head = make_head(tuned_checkpoint["dropout"], tuned_checkpoint["state_dict"])
print("Selected universal:", universal_path, "Zn validation median MSE:", universal_result["zn_val_median_mse"])
print("Selected tuned:", tuned_path, "Zn validation median MSE:", tuned_result["zn_val_median_mse"], "selection:", tuned_result["selection"])


## Evaluate Zn test spectra

The baseline is the mean Zn training spectrum. Eta is the median per-spectrum MSE of that baseline divided by the median per-spectrum MSE of the head. Higher eta is better.


In [ ]:
test_ids, test_X, test_y = parts["test"]
train_mean = parts["train"][2].mean(axis=0)
with torch.inference_mode():
    predictions = torch.cat([head(batch.to(device)).cpu() for batch in torch.from_numpy(test_X).split(BATCH_SIZE)]).numpy()
if predictions.shape != test_y.shape or not np.isfinite(predictions).all():
    raise ValueError("Invalid test predictions.")
errors = ((predictions - test_y) ** 2).mean(axis=1)
baseline_errors = ((test_y - train_mean) ** 2).mean(axis=1)
median_mse = float(np.median(errors))
baseline_median = float(np.median(baseline_errors))
eta = baseline_median / median_mse if median_mse else (float("inf") if baseline_median else float("nan"))
print(f"310–500–750 | test spectra: {len(test_ids)} | MSE: {errors.mean():.6g} | median MSE: {median_mse:.6g} | eta: {eta:.3f}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.savez_compressed(OUTPUT_DIR / "test_predictions.npz", predictions=predictions, targets=test_y, ids=np.asarray(test_ids))
(OUTPUT_DIR / "test_metrics.json").write_text(json.dumps({"hidden_dims": DIMS, "head_sha256": sha256(HEAD_DIR / "best.pt"),
    "encoder_sha256": provenance["encoder_sha256"], "dataset_sha256": dataset_hash,
    "test_count": len(test_ids), "mse": float(errors.mean()), "median_mse": median_mse,
    "baseline_median_mse": baseline_median, "eta": eta}, indent=2), encoding="utf-8")


## Compare the three heads on the same Zn test spectra

The table uses the same test rows and Zn training-mean baseline for all three heads. Checkpoints are already selected using validation data. The test set and iodine example have been inspected before, so this is an exploratory comparison.


In [ ]:
model_heads = {"ExpertXAS": head, "UniversalXAS": universal_head, "Tuned-UniversalXAS": tuned_head}
model_predictions = {"ExpertXAS": predictions}
model_predictions.update({name: predict_head(model, test_X) for name, model in model_heads.items() if name != "ExpertXAS"})
model_errors = {name: ((pred - test_y) ** 2).mean(axis=1) for name, pred in model_predictions.items()}
comparison = []
for name, values in model_errors.items():
    med = float(np.median(values))
    comparison.append({"model": name, "test_mse": float(values.mean()), "test_median_mse": med,
                       "test_eta": baseline_median / med if med else (float("inf") if baseline_median else float("nan"))})
comparison = pd.DataFrame(comparison)
display(comparison)
comparison.to_csv(OUTPUT_DIR / "three_head_test_metrics.csv", index=False)
np.savez_compressed(OUTPUT_DIR / "three_head_test_predictions.npz", ids=np.asarray(test_ids), targets=test_y,
                    expert=model_predictions["ExpertXAS"], universal=model_predictions["UniversalXAS"], tuned=model_predictions["Tuned-UniversalXAS"])


## Test error and spectrum detail

Each dot is one **test** spectrum. Farther right means a larger prediction error. Higher up means the target spectrum has more sharp or rapidly changing detail. The right panel zooms in on most dots. The red dot marks the largest error.

The detail ratio compares each target with a Gaussian-smoothed version (sigma 0.5 eV). Sharp detail can be a real spectral feature, not noise. Both correlations use all test spectra; correlation does not prove a cause.


In [ ]:
from scipy.stats import pearsonr, spearmanr

def noise_ratio(y):
    scale = float(np.std(y))
    if scale == 0:
        return 0.0
    sigma_points = SIGMA_EV / (ENERGY_GRID[1] - ENERGY_GRID[0])
    half = max(1, int(3 * sigma_points))
    x = np.arange(-half, half + 1)
    kernel = np.exp(-0.5 * (x / sigma_points) ** 2)
    kernel /= kernel.sum()
    smooth = np.convolve(np.pad(y, half, mode="edge"), kernel, mode="valid")
    return float(np.std(y - smooth) / scale)

ratios = np.array([noise_ratio(y) for y in test_y])
pearson = float(pearsonr(errors, ratios).statistic) if np.ptp(errors) and np.ptp(ratios) else float("nan")
spearman = float(spearmanr(errors, ratios).statistic) if np.ptp(errors) and np.ptp(ratios) else float("nan")
chart = pd.DataFrame({"id": test_ids, "mse": errors, "noise_ratio": ratios})
chart[["material_id", "site"]] = chart["id"].str.rsplit("_", n=1, expand=True)
chart.to_csv(OUTPUT_DIR / "test_noise_ratio.csv", index=False)
worst = int(np.argmax(errors))
xmax = max(float(np.percentile(errors, 99)), np.finfo(float).eps)
ymax = max(float(np.percentile(ratios, 99)), np.finfo(float).eps)
visible = (errors <= xmax) & (ratios <= ymax)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax in axes:
    ax.scatter(errors, ratios, s=18, alpha=0.6)
    ax.set_xlabel("Per-spectrum MSE (test)")
    ax.set_ylabel("High-frequency ratio (Gaussian sigma = 0.5 eV)")
    ax.grid(alpha=0.3)
axes[0].scatter(errors[worst], ratios[worst], s=55, color="tab:red")
axes[0].annotate(f"Zn {chart.material_id.iloc[worst]} site {chart.site.iloc[worst]}",
                 (errors[worst], ratios[worst]), xytext=(-8, -12), textcoords="offset points", ha="right", fontsize=8)
axes[0].set_title(f"All {len(errors)} spectra | Pearson r = {pearson:.2f} | Spearman rho = {spearman:.2f}")
axes[1].set_xlim(0, xmax)
axes[1].set_ylim(0, ymax)
axes[1].set_title(f"99th-percentile zoom | {int(visible.sum())} of {len(errors)} shown")
fig.suptitle("Zn test: 310–500–750")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "mse_vs_noise.png", dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
display(chart.sort_values("mse", ascending=False).head(5))


## Compare errors and the iodine outlier

Each column uses the same test spectra and target-derived detail ratios. The first row shows all test points. The second zooms to common limits based on the pooled 99th-percentile error and the 99th-percentile detail ratio. The red point is Zn mp-571292 site 000. A separate plot overlays the three predictions for that spectrum. Interpret these plots descriptively, not as a basis for choosing a model.


In [ ]:
outlier_id = "mp-571292_000"
if outlier_id not in test_ids:
    raise ValueError(f"Expected held-out iodine spectrum not found: {outlier_id}")
outlier = test_ids.index(outlier_id)
names = list(model_errors)
pooled = np.concatenate([model_errors[name] for name in names])
xmax = max(float(np.percentile(pooled, 99)), np.finfo(float).eps)
ymax = max(float(np.percentile(ratios, 99)), np.finfo(float).eps)
fig, axes = plt.subplots(2, 3, figsize=(17, 9), sharey="row")
for column, name in enumerate(names):
    err = model_errors[name]
    for row in range(2):
        ax = axes[row, column]
        ax.scatter(err, ratios, s=14, alpha=0.5)
        ax.scatter(err[outlier], ratios[outlier], s=55, c="red", zorder=3)
        ax.set_xlabel("Per-spectrum test MSE")
        ax.grid(alpha=0.2)
    axes[0, column].set_title(name)
    axes[0, column].set_xlim(0, float(pooled.max()) * 1.05)
    axes[1, column].set_xlim(0, xmax)
    axes[1, column].set_ylim(0, ymax)
axes[0, 0].set_ylabel("Detail ratio — all test spectra")
axes[1, 0].set_ylabel("Detail ratio — zoom")
fig.suptitle("Zn test: red marks mp-571292 site 000")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "three_head_scatter.png", dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ENERGY_GRID, test_y[outlier], color="black", lw=2, label="Target")
for name, pred in model_predictions.items():
    ax.plot(ENERGY_GRID, pred[outlier], label=f"{name} (MSE {model_errors[name][outlier]:.4g})")
ax.set_title("Zn mp-571292 site 000 — held-out test spectrum")
ax.set_xlabel("Energy (eV)")
ax.set_ylabel("Intensity (unscaled)")
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "iodine_outlier_three_heads.png", dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)


## Test spectra by MSE percentile

The first grid shows spectra near nine MSE percentiles. The second shows nine distinct materials near the 95th percentile. Labels include the element, material ID, absorbing site, and MSE.


In [ ]:
percentiles = [5, 10, 20, 30, 50, 70, 80, 90, 95]
selected = []
for q in percentiles:
    order = np.argsort(abs(errors - np.percentile(errors, q)), kind="stable")
    i = next((int(index) for index in order if int(index) not in selected), None)
    if i is None:
        raise ValueError("At least nine test spectra are needed.")
    selected.append(i)

def spectrum_grid(indices, title, filename, percentile_labels=None):
    fig, axes = plt.subplots(3, 3, figsize=(14, 10), sharex=True)
    for position, (ax, i) in enumerate(zip(axes.flat, indices)):
        ax.plot(ENERGY_GRID, test_y[i], label="Target")
        ax.plot(ENERGY_GRID, predictions[i], label="Prediction")
        mid, site = test_ids[i].rsplit("_", 1)
        percentile = f" | {percentile_labels[position]}th percentile" if percentile_labels is not None else ""
        ax.set_title(f"Zn {mid} site {site}{percentile}\nMSE {errors[i]:.4g}", fontsize=9)
        ax.set_xlabel("Energy (eV)")
        ax.set_ylabel("Intensity (unscaled)")
        ax.grid(alpha=0.2)
    axes.flat[0].legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, dpi=180)
    display(fig)
    plt.close(fig)

spectrum_grid(selected, "Zn 310–500–750: test MSE percentiles", "test_percentiles.png", percentiles)
q95 = float(np.percentile(errors, 95))
near95, seen = [], set()
for i in np.argsort(abs(errors - q95), kind="stable"):
    mid = test_ids[i].rsplit("_", 1)[0]
    if mid not in seen:
        seen.add(mid)
        near95.append(int(i))
    if len(near95) == 9:
        break
if len(near95) < 9:
    raise ValueError("At least nine distinct test materials are needed.")
spectrum_grid(near95, f"Zn 310–500–750: distinct materials near 95th-percentile MSE ({q95:.4g})", "test_near_p95.png")


## Iodine-containing Zn spectra

The main metrics and plots above use the **test** split. This iodine comparison also includes training spectra so you can compare their shapes. Each panel labels its split. **TEST** means the spectrum was held out of training; **TRAIN** means the model trained on it.

Each title shows baseline MSE divided by model MSE for that spectrum. This is not the paper's dataset-level eta. Source `structure.json` files must be available.


In [ ]:
iodine = []
for split in SPLITS:
    ids, X, y = parts[split]
    selected = []
    for i, value in enumerate(ids):
        mid, site = value.rsplit("_", 1)
        path = RAW_ROOT / "FEFF" / "Zn" / mid / "FEFF-XANES" / f"{int(site):03d}_Zn" / "structure.json"
        if not path.is_file():
            raise FileNotFoundError(f"Missing source structure: {path}")
        structure = json.loads(path.read_text(encoding="utf-8"))
        if any(species.get("element") == "I" for atom in structure["sites"] for species in atom["species"]):
            selected.append(i)
    if not selected:
        continue
    with torch.inference_mode():
        batches = torch.from_numpy(X[selected]).split(BATCH_SIZE)
        split_predictions = torch.cat([head(batch.to(device)).cpu() for batch in batches]).numpy()
    for i, prediction in zip(selected, split_predictions):
        target = y[i]
        model_mse = float(np.mean((target - prediction) ** 2))
        baseline_mse = float(np.mean((target - train_mean) ** 2))
        ratio = baseline_mse / model_mse if model_mse else (float("inf") if baseline_mse else float("nan"))
        iodine.append((ids[i], split, target, prediction, ratio))

iodine.sort(key=lambda item: {"test": 0, "val": 1, "train": 2}[item[1]])

if not iodine:
    print("No iodine-containing Zn spectra found.")
else:
    print("Iodine-containing Zn spectra by split:", pd.Series([row[1] for row in iodine]).value_counts().to_dict())
    fig, axes = plt.subplots(math.ceil(len(iodine) / 3), 3, figsize=(15, 3.8 * math.ceil(len(iodine) / 3)), squeeze=False, sharex=True)
    for ax, (value, split, target, prediction, ratio) in zip(axes.flat, iodine):
        ax.plot(ENERGY_GRID, target, label="Target")
        ax.plot(ENERGY_GRID, prediction, label="Prediction")
        mid, site = value.rsplit("_", 1)
        split_label = {"test": "TEST — held out", "train": "TRAIN — used for training", "val": "VALIDATION — held out"}[split]
        ax.set_title(f"Zn {mid} site {site} | {split_label}\nBaseline MSE / model MSE = {ratio:.2f}",
                     color="darkred" if split == "test" else "black", fontweight="bold" if split == "test" else "normal", fontsize=9)
        ax.set_xlabel("Energy (eV)")
        ax.set_ylabel("Intensity (unscaled)")
        ax.grid(alpha=0.2)
    for ax in axes.flat[len(iodine):]:
        ax.set_visible(False)
    axes.flat[0].legend()
    fig.suptitle("Iodine-containing Zn spectra: test and training comparison | 310–500–750")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "iodine_spectra.png", dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
